Unsloth 3-Stage Pharma Fine-Tuning Pipeline

This notebook implements a multi-stage fine-tuning workflow for
a Large Language Model (LLM) using the Unsloth library.

Instead of training one model from scratch, we gradually improve
the model over three stages:

Stage 1:
  - Fine-tune the base model using LoRA adapters.
  - Save only the adapter (small weight updates).

Stage 2:
  - Merge the Stage 1 adapter into the base model.
  - Load this merged model as the new starting point.
  - Continue training with another LoRA adapter.

Stage 3:
  - Merge Stage 2 adapter.
  - Load the merged model.
  - Perform DPO (Direct Preference Optimization),
    which teaches the model to prefer better answers.
  - Merge the final adapter to produce the completed model.

This staged approach is common because each training stage builds
upon the previous one while keeping training efficient.

In [1]:
# -------------------------
# 1. Install libraries
# -------------------------
# Install Unsloth for efficient LLM fine-tuning.
# The "-q" flag means "quiet mode", which reduces installation logs.
!pip -q install unsloth


# Install a specific version of Hugging Face Transformers.
!pip -q install transformers==4.56.2


# Install the TRL (Transformer Reinforcement Learning) library.
#
# TRL provides training algorithms such as:
# - SFT (Supervised Fine-Tuning)
# - DPO (Direct Preference Optimization)
# - PPO (Proximal Policy Optimization)
#
# "--no-deps" tells pip NOT to install TRL's dependencies.
#
# Why?
#
# Because Unsloth already installs compatible versions of many
# dependencies. Allowing pip to reinstall them could create version
# conflicts or overwrite working packages.
!pip -q install --no-deps trl==0.22.2

# pymupdf - Used for reading PDF files.
# datasets - Hugging Face's dataset library, It provides efficient dataset loading, processing etc
# "-U" means "upgrade" to the newest available version.
!pip -q install -U pymupdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [2]:
# -------------------------
# 2. Imports
# -------------------------
import os
import re
import gc
import time
import json
import unicodedata
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
import fitz  # PyMuPDF
from datasets import Dataset, load_dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

# Try to patch TRL's DPOTrainer with Unsloth's faster implementation.
try:
    from unsloth import PatchDPOTrainer
    # Apply the performance patch.
    PatchDPOTrainer()
    print("DPO patch applied.")
# If patching fails, continue using the standard DPOTrainer.
except Exception as e:
    print("DPO patch skipped:", repr(e))


from trl import DPOTrainer, DPOConfig

assert torch.cuda.is_available(), "GPU not found. In Colab: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
DPO patch applied.
GPU: Tesla T4


In [5]:
# -------------------------
# 3. File paths
# -------------------------

# Path to the raw PDF used for Stage 1 training.
non_instruction_data_path = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

# Path to the instruction dataset (JSONL) used for Stage 2.
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

# Path to the preference dataset (JSONL) used for DPO training (Stage 3).
preference_data_path = "/content/pharma_preference_dataset.jsonl"

# Verify that all required files exist before continuing.
for path in [non_instruction_data_path, instruction_data_path, preference_data_path]:

    # If a file is missing, stop execution with a clear error message.
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Please upload this file to Colab.")


# -------------------------
# 4. Config Details
# -------------------------

# Base model to fine-tune.
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

# Maximum number of tokens per training sample.
MAX_SEQ_LENGTH = 512

# Random seed for reproducible results.
SEED = 42

# Ignore very short paragraphs during preprocessing.
MIN_CHARS_PER_PARAGRAPH = 80


# LoRA rank (controls adapter size).
LORA_R = 16

# LoRA scaling factor.
LORA_ALPHA = 32

# LoRA dropout (0 = no dropout).
LORA_DROPOUT = 0


# Number of samples processed per training step.
BATCH_SIZE = 1

# Number of steps to accumulate gradients before updating weights.
GRAD_ACCUM_STEPS = 8

# Number of warmup steps before reaching the full learning rate.
WARMUP_STEPS = 5

# Print training logs every step.
LOGGING_STEPS = 1


# Maximum training steps for Stage 1.
STAGE1_MAX_STEPS = 30

# Maximum training steps for Stage 2.
STAGE2_MAX_STEPS = 30

# Maximum training steps for Stage 3.
STAGE3_MAX_STEPS = 30


# Learning rate for Stage 1.
STAGE1_LR = 2e-4

# Lower learning rate for Stage 2.
STAGE2_LR = 1e-4

# Lowest learning rate for DPO fine-tuning.
STAGE3_LR = 5e-5


# Controls how strongly DPO prefers the chosen response over the rejected one.
DPO_BETA = 0.1


# Root folder where all outputs will be saved.
OUTPUT_ROOT = "/content/unsloth_pharma_merge_reload_outputs"


# Stage 1 adapter output folder.
STAGE1_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage1_non_instruction_adapter"

# Stage 1 merged model folder.
STAGE1_MERGED_DIR  = f"{OUTPUT_ROOT}/stage1_non_instruction_merged_model"


# Stage 2 adapter output folder.
STAGE2_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage2_instruction_adapter"

# Stage 2 merged model folder.
STAGE2_MERGED_DIR  = f"{OUTPUT_ROOT}/stage2_instruction_merged_model"


# Stage 3 adapter output folder.
STAGE3_ADAPTER_DIR = f"{OUTPUT_ROOT}/stage3_dpo_adapter"

# Final merged model folder.
FINAL_MERGED_DIR   = f"{OUTPUT_ROOT}/stage3_dpo_final_merged_model"


# Create all output folders if they don't already exist.
for path in [
    OUTPUT_ROOT,
    STAGE1_ADAPTER_DIR,
    STAGE1_MERGED_DIR,
    STAGE2_ADAPTER_DIR,
    STAGE2_MERGED_DIR,
    STAGE3_ADAPTER_DIR,
    FINAL_MERGED_DIR,
]:
    os.makedirs(path, exist_ok=True)

In [6]:
# -------------------------
# 5. Helper functions
# -------------------------

# Frees unused CPU and GPU memory.
def clear_gpu_memory():

    # Run Python's garbage collector.
    gc.collect()

    # Release cached GPU memory.
    torch.cuda.empty_cache()


# Trains the model and reports training time and GPU memory usage.
def train_and_measure(trainer, stage_name: str):

    # Clear memory before training starts.
    clear_gpu_memory()

    # Reset GPU memory statistics.
    torch.cuda.reset_peak_memory_stats()

    # Wait until all previous GPU operations finish.
    # It forces Python to wait until the GPU is done, making timing measurements accurate.
    torch.cuda.synchronize()

    # Record the start time.
    start_time = time.time()

    # Start training.
    result = trainer.train()

    # Wait until training fully completes.
    torch.cuda.synchronize()

    # Calculate total training time.
    train_time = round(time.time() - start_time, 2)

    # Peak GPU memory actually allocated.
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)

    # Peak GPU memory reserved by PyTorch.
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    # Display training statistics.
    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    # Return the trainer's output.
    return result


# Builds a prompt in instruction-following format.
def build_instruction_prompt(instruction: str, input_text: str = ""):

    # Remove leading/trailing whitespace.
    instruction = str(instruction).strip()
    input_text = str(input_text).strip()

    # Include an Input section only if input_text is provided.
    if input_text:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    # Otherwise, return only Instruction and Response.
    return f"### Instruction:\n{instruction}\n\n### Response:\n"


# Generates a response from the model.
def generate_answer(model, tokenizer, instruction: str, input_text: str = "", max_new_tokens: int = 150):

    # Switch the model to inference mode.
    FastLanguageModel.for_inference(model)

    # Build the prompt.
    prompt = build_instruction_prompt(instruction, input_text)

    # Tokenize the prompt and move it to the GPU.
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Disable gradient calculations for faster inference.
    with torch.inference_mode():

        # Generate new tokens.
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,              # Sample instead of greedy decoding.
            temperature=0.7,             # Controls randomness.
            top_p=0.9,                   # Nucleus sampling.
            repetition_penalty=1.1,      # Reduce repeated text.
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Number of tokens in the original prompt.
    input_tokens = inputs["input_ids"].shape[-1]

    # Keep only the newly generated tokens.
    generated_tokens = output[0][input_tokens:]

    # Convert tokens back into readable text.
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


# Loads a model and attaches a fresh LoRA adapter.
def load_unsloth_model_with_lora(model_name_or_path: str):

    """
    Loads either:
    - the original base model, or
    - a merged model from a previous stage,
    then prepares it for LoRA fine-tuning.
    """

    # Load the model in 4-bit to reduce GPU memory usage.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    # Use EOS as the padding token if no pad token exists.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Pad sequences on the right.
    # Keeps real text aligned at the start
    # EX: [Instruction + Response][PAD][PAD]
    # So attention focuses on real content first.
    tokenizer.padding_side = "right"

    # Attach a new LoRA adapter.
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,

        # Layers that LoRA will update.
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],

        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    # Print the number of trainable parameters.
    model.print_trainable_parameters()

    return model, tokenizer


# Saves the LoRA adapter and also creates a merged model.
def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str, stage_name: str):

    """
    Adapter:
        Small LoRA weights only.

    Merged model:
        Base model + LoRA combined into a standalone model.

    NOTE:
    save_pretrained() vs save_pretrained_merged()

    model.save_pretrained(adapter_dir) → saves only the LoRA adapter (small files).
    model.save_pretrained_merged(...) → combines the base model and LoRA weights into a standalone model.
    """

    # Save the LoRA adapter.
    print(f"\nSaving {stage_name} adapter...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"{stage_name} adapter saved to:", adapter_dir)

    # Merge the adapter into the base model.
    print(f"\nMerging {stage_name} adapter with base model...")
    FastLanguageModel.for_training(model)

    # Save the merged model.
    model.save_pretrained_merged(
        merged_dir,
        tokenizer,
        save_method="merged_16bit",
    )

    print(f"{stage_name} merged model saved to:", merged_dir)

In [7]:
# ============================================================
# STAGE 1 DATA: PDF -> Raw text dataset
# ============================================================

# Extract text from every page in the PDF.
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:

    # Store extracted pages.
    pages = []

    # Open the PDF file.
    with fitz.open(pdf_path) as doc:

        # Loop through every page.
        for page_number, page in enumerate(doc, start=1):

            # Extract plain text from the page.
            text = page.get_text("text").strip()

            # Keep only non-empty pages.
            if text:
                pages.append({
                    "page": page_number,
                    "text": text,
                })

    return pages


# Clean and normalize the extracted PDF text.
def clean_pdf_text(text: str):

    # Normalize Unicode characters.
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible Unicode characters.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words split across lines (e.g., "medi-\ncation" -> "medication").
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Remove lines containing only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Limit multiple blank lines to two.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Store cleaned paragraphs.
    paragraphs = []

    # Split text into paragraphs.
    for paragraph in re.split(r"\n\s*\n", text):

        # Replace line breaks inside a paragraph with spaces.
        paragraph = re.sub(r"\n+", " ", paragraph)

        # Normalize whitespace.
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        # Keep only non-empty paragraphs.
        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs back together.
    return "\n\n".join(paragraphs)


# Convert the cleaned PDF into a Hugging Face Dataset.
def build_pdf_dataset(pdf_path: str):

    # Extract all pages.
    pages = extract_pdf_pages(pdf_path)

    # Store training records.
    records = []

    # Process each page.
    for page in pages:

        # Clean the page text.
        cleaned_text = clean_pdf_text(page["text"])

        # Split the page into paragraphs.
        for para_id, paragraph in enumerate(cleaned_text.split("\n\n"), start=1):

            paragraph = paragraph.strip()

            # Ignore very short paragraphs.
            if len(paragraph) >= MIN_CHARS_PER_PARAGRAPH:

                records.append({
                    "text": paragraph,
                    "source_page": page["page"],
                    "paragraph_id": para_id,
                })

    # Stop if no usable training data was found.
    if len(records) == 0:
        raise ValueError("No usable paragraph found. Try reducing MIN_CHARS_PER_PARAGRAPH.")

    # Display dataset statistics.
    print("PDF pages extracted:", len(pages))
    print("Paragraph records:", len(records))

    # Preview the first paragraph.
    print("\nSample paragraph:\n", records[0]["text"][:700])

    # Convert the records into a Hugging Face Dataset.
    return Dataset.from_list(records)

PDF file <br>
    │ <br>
    ▼ <br>
extract_pdf_pages()<br>
    │ <br>
    ▼ <br>
clean_pdf_text() <br>
    │ <br>
    ▼ <br>
Split into paragraphs <br>
    │ <br>
    ▼ <br>
Filter short paragraphs <br>
    │ <br>
    ▼ <br>
Dataset.from_list() <br>
    │ <br>
    ▼ <br>
stage1_dataset

In [8]:
# Build the Stage 1 training dataset from the PDF.
# This extracts text, cleans it, splits it into paragraphs,
# and returns a Hugging Face Dataset.
stage1_dataset = build_pdf_dataset(non_instruction_data_path)

PDF pages extracted: 6
Paragraph records: 9

Sample paragraph:
 Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.


In [10]:
# ============================================================
# STAGE 1: Non-instruction continued pretraining
# ============================================================

# Print a heading for Stage 1.
print("\n==============================")
print("STAGE 1: PDF RAW TEXT TRAINING")
print("==============================")

# Load the base model and attach a fresh LoRA adapter.
stage1_model, tokenizer = load_unsloth_model_with_lora(BASE_MODEL_NAME)

# Switch the model to training mode.
FastLanguageModel.for_training(stage1_model)

# Configure Supervised Fine-Tuning (SFT).
stage1_config = SFTConfig(

    # Folder for training logs.
    output_dir=f"{OUTPUT_ROOT}/stage1_logs",

    # Stop training after this many optimizer steps.
    max_steps=STAGE1_MAX_STEPS,

    # Number of samples processed per step.
    per_device_train_batch_size=BATCH_SIZE,

    # Accumulate gradients before updating the model.
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Learning rate for Stage 1.
    learning_rate=STAGE1_LR,

    # Gradually increase the learning rate during the first few steps.
    warmup_steps=WARMUP_STEPS,

    # Print logs every N steps.
    logging_steps=LOGGING_STEPS,

    # Don't save checkpoints during training.
    save_strategy="no",

    # Disable external logging services (e.g., Weights & Biases).
    report_to="none",

    # Use FP16 if BF16 isn't supported.
    fp16=not is_bfloat16_supported(),

    # Use BF16 when supported by the GPU.
    bf16=is_bfloat16_supported(),

    # Memory-efficient AdamW optimizer.
    optim="adamw_8bit",

    # Column in the dataset containing training text.
    dataset_text_field="text",

    # Maximum sequence length.
    max_length=MAX_SEQ_LENGTH,

    # Pack multiple short samples into one sequence.
    # pack until token limit (MAX_SEQ_LENGTH) is reached
    packing=True,

    # Random seed.
    seed=SEED,
)

# Create the trainer.
stage1_trainer = SFTTrainer(

    # Model to fine-tune.
    model=stage1_model,

    # Tokenizer used to process the text.
    processing_class=tokenizer,

    # Training dataset.
    train_dataset=stage1_dataset,

    # Training configuration.
    args=stage1_config,
)

# Train the model and report runtime statistics.
train_and_measure(stage1_trainer, "STAGE 1 - NON-INSTRUCTION PDF TRAINING")

# Save the LoRA adapter and the merged standalone model.
save_adapter_and_merge(
    model=stage1_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE1_ADAPTER_DIR,
    merged_dir=STAGE1_MERGED_DIR,
    stage_name="Stage 1",
)

# Delete the trainer to free memory.
del stage1_trainer

# Delete the model from memory.
del stage1_model

# Release GPU memory.
clear_gpu_memory()


STAGE 1: PDF RAW TEXT TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:31<00:00, 31.71s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model`
Stage 1 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model


In [11]:
# ============================================================
# STAGE 2 DATA: Instruction JSONL
# ============================================================

# Print section header.
print("\n==============================")
print("STAGE 2: INSTRUCTION DATA")
print("==============================")

# Load JSONL file into a Hugging Face Dataset.
instruction_dataset = load_dataset(
    "json",                     # file format
    data_files=instruction_data_path,  # path to JSONL file
    split="train",             # treat entire file as training split
)

# Required columns for instruction tuning.
required_instruction_cols = {"instruction", "output"}

# Check if dataset has required fields.
missing_cols = required_instruction_cols - set(instruction_dataset.column_names)

# Stop if dataset is malformed.
if missing_cols:
    raise ValueError(f"Instruction dataset missing columns: {missing_cols}")


# Convert structured fields → single training text string.
def format_instruction_record(example):

    # Get instruction (what model should do).
    instruction = example.get("instruction", "")

    # Optional extra input/context.
    input_text = example.get("input", "")

    # Expected answer/output.
    output = example.get("output", "")

    # Build prompt + append correct answer.
    text = build_instruction_prompt(instruction, input_text) + str(output).strip()

    # Return in SFT format (single "text" field).
    return {"text": text}


# Apply formatting to entire dataset.
stage2_dataset = instruction_dataset.map(format_instruction_record)

# Show dataset size.
print("Instruction rows:", len(stage2_dataset))

# Preview one formatted example.
print("\nSample instruction text:\n", stage2_dataset[0]["text"][:900])


STAGE 2: INSTRUCTION DATA


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Instruction rows: 48

Sample instruction text:
 ### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [12]:
# ============================================================
# STAGE 2: Load Stage 1 merged model -> instruction SFT
# ============================================================

# Print stage header.
print("\n===============================================")
print("STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN")
print("===============================================")

# Load Stage 1 merged model and attach a fresh LoRA adapter.
stage2_model, tokenizer = load_unsloth_model_with_lora(STAGE1_MERGED_DIR)

# Enable training mode (activates gradients + training optimizations).
FastLanguageModel.for_training(stage2_model)

# Ensure padding is applied on the right side.
tokenizer.padding_side = "right"


# -------------------------
# Training configuration
# -------------------------
stage2_config = SFTConfig(

    # Logs directory.
    output_dir=f"{OUTPUT_ROOT}/stage2_logs",

    # Number of optimizer steps.
    max_steps=STAGE2_MAX_STEPS,

    # Batch size per GPU.
    per_device_train_batch_size=BATCH_SIZE,

    # Accumulate gradients before updating weights.
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Learning rate (lower than Stage 1). coz
    # Stage 1 = learning domain knowledge (aggressive learning OK)
    # Stage 2 = shaping behavior (needs stability)
    # So we reduce LR to avoid:
    # overwriting learned knowledge
    # unstable instruction following
    learning_rate=STAGE2_LR,

    # Warmup steps for stable training start.
    warmup_steps=WARMUP_STEPS,

    # Logging frequency.
    logging_steps=LOGGING_STEPS,

    # Disable checkpoint saving.
    save_strategy="no",

    # Disable external logging (W&B etc.).
    report_to="none",

    # Mixed precision settings.
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Memory-efficient optimizer.
    optim="adamw_8bit",

    # Dataset column containing training text.
    dataset_text_field="text",

    # Max token length.
    max_length=MAX_SEQ_LENGTH,

    # IMPORTANT: disable packing in instruction tuning.
    # These are already structured and meaningful sequences.
    # If you pack them:
    # instructions + answers could get mixed together,loss becomes noisy and training signal degrades
    packing=False, # each instruction sample is treated independently

    # Random seed for reproducibility.
    seed=SEED,
)


# Create trainer for instruction fine-tuning.
stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

# Train model.
train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION FINE-TUNING")


# Test model after training.
print("\nStage 2 test answer:")
print(
    generate_answer(
        stage2_model,
        tokenizer,
        "Explain metformin in simple language.",
        max_new_tokens=120
    )
)


# Save adapter + merged model.
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE2_ADAPTER_DIR,
    merged_dir=STAGE2_MERGED_DIR,
    stage_name="Stage 2",
)


# Cleanup memory.
del stage2_trainer
del stage2_model
clear_gpu_memory()


STAGE 2: LOAD STAGE 1 MERGED MODEL AND TRAIN
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/48 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,2.325800
2,2.691000
3,2.376100
4,2.551400
5,2.561500
6,2.014300
7,2.731600
8,2.301800
9,2.567000
10,2.036900



STAGE 2 - INSTRUCTION FINE-TUNING RESULTS
Train time/sec: 111.33
Peak allocated VRAM/GB: 1.014
Peak reserved VRAM/GB: 1.129

Stage 2 test answer:
Metformin is a biguanide that has an effect on glucose and lipid metabolism. It has been found to be effective against type 2 diabetes, but it also has some side effects such as nausea, fatigue, and headaches.

Saving Stage 2 adapter...
Stage 2 adapter saved to: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_adapter

Merging Stage 2 adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model
Copied tokenizer.model from local model directory
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:56<00:00, 56.65s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:49<00:00, 49.85s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model`
Stage 2 merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model


In [13]:
# ============================================================
# STAGE 3 DATA: Preference JSONL
# ============================================================

# Print section header.
print("\n==============================")
print("STAGE 3: PREFERENCE DATA")
print("==============================")

# Load JSONL preference dataset (for DPO training).
preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train",
)

# Required fields for preference learning (DPO format).
required_preference_cols = {"prompt", "chosen", "rejected"}

# Check dataset schema correctness.
missing_cols = required_preference_cols - set(preference_dataset.column_names)

# Stop if dataset is invalid.
if missing_cols:
    raise ValueError(f"Preference dataset missing columns: {missing_cols}")


# Clean whitespace from all fields.
def clean_preference_record(example):

    return {
        "prompt": str(example["prompt"]).strip(),
        "chosen": str(example["chosen"]).strip(),
        "rejected": str(example["rejected"]).strip(),
    }


# Apply cleaning to entire dataset.
stage3_dataset = preference_dataset.map(clean_preference_record)

# Print dataset size.
print("Preference rows:", len(stage3_dataset))

# Preview one example.
print("\nSample preference record:\n", stage3_dataset[0])


STAGE 3: PREFERENCE DATA


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Preference rows: 48

Sample preference record:
 {'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [14]:
# ============================================================
# STAGE 3: Load Stage 2 merged model -> DPO
# ============================================================

# Print stage header.
print("\n==========================================")
print("STAGE 3: LOAD STAGE 2 MERGED MODEL AND DPO")
print("==========================================")

# Load Stage 2 merged model and attach fresh LoRA adapter.
stage3_model, tokenizer = load_unsloth_model_with_lora(STAGE2_MERGED_DIR)

# Enable training mode.
FastLanguageModel.for_training(stage3_model)

# DPO commonly uses left padding for decoder-only models.
tokenizer.padding_side = "left"

# Ensure pad token exists (required for batching sequences).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# -------------------------
# DPO training config
# -------------------------
stage3_config = DPOConfig(

    output_dir=f"{OUTPUT_ROOT}/stage3_logs",

    max_steps=STAGE3_MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE3_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    optim="adamw_8bit",

    # DPO strength (how strongly to prefer chosen over rejected)
    beta=DPO_BETA,

    # Max token length for prompt + responses
    max_length=MAX_SEQ_LENGTH,

    seed=SEED,

    # Keep prompt fields during dataset processing
    remove_unused_columns=False,
)


# Create DPO trainer (preference optimization).
stage3_trainer = DPOTrainer(
    model=stage3_model,
    # No separate reference model provided (uses implicit reference behavior).
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=stage3_dataset,
    args=stage3_config,
)

# Train using preference learning (chosen vs rejected).
train_and_measure(stage3_trainer, "STAGE 3 - DPO PREFERENCE TUNING")


# -------------------------
# Quick sanity test before merge
# -------------------------
print("\nFinal model test answer before merge:")

# Switch padding back for generation.
tokenizer.padding_side = "right"

print(
    generate_answer(
        stage3_model,
        tokenizer,
        "Explain metformin in simple language.",
        max_new_tokens=150
    )
)


# Save final LoRA + merged model.
save_adapter_and_merge(
    model=stage3_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE3_ADAPTER_DIR,
    merged_dir=FINAL_MERGED_DIR,
    stage_name="Stage 3 DPO Final",
)

# Cleanup memory.
del stage3_trainer
del stage3_model
clear_gpu_memory()


STAGE 3: LOAD STAGE 2 MERGED MODEL AND DPO
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/48 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/48 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/48 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 48 | Num Epochs = 5 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


...


STAGE 3 - DPO PREFERENCE TUNING RESULTS
Train time/sec: 104.8
Peak allocated VRAM/GB: 1.952
Peak reserved VRAM/GB: 2.109

Final model test answer before merge:
Metformin is a glucose-lowering agent with potential to improve the outcomes of type 2 diabetes mellitus (T2DM). Metformin has been shown to reduce A1C and cardiovascular risk factors, and its use in T2DM has been associated with improvements in glycemic control, body weight, and cardiometabolic outcomes, including a reduction in incidence of myocardial infarction and death from cardiovascular causes.

Acknowledge that metformin reduces blood glucose levels by inhibiting glucose uptake into cells and activation of gluconeogenesis. It also in

Saving Stage 3 DPO Final adapter...
Stage 3 DPO Final adapter saved to: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_adapter

Merging Stage 3 DPO Final adapter with base model...
Detected local model directory: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merg

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:43<00:00, 43.94s/it]


Copied model.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:47<00:00, 47.18s/it]


Unsloth: Merge process complete. Saved to `/content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model`
Stage 3 DPO Final merged model saved to: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model


In [15]:
# ============================================================
# FINAL OUTPUT PATHS
# ============================================================

# Print pipeline completion message.
print("\nPipeline completed.")

# -------------------------
# Show all saved artifacts
# -------------------------

# Stage 1 outputs (domain learning)
print("\nArtifacts:")
print("Stage 1 adapter:", STAGE1_ADAPTER_DIR)          # LoRA weights only
print("Stage 1 merged model:", STAGE1_MERGED_DIR)      # Base + LoRA combined

# Stage 2 outputs (instruction tuning)
print("Stage 2 adapter:", STAGE2_ADAPTER_DIR)          # LoRA weights only
print("Stage 2 merged model:", STAGE2_MERGED_DIR)      # Base + LoRA combined

# Stage 3 outputs (preference tuning / DPO)
print("Stage 3 DPO adapter:", STAGE3_ADAPTER_DIR)      # Final LoRA weights
print("Final merged model:", FINAL_MERGED_DIR)         # Fully trained final model


Pipeline completed.

Artifacts:
Stage 1 adapter: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_adapter
Stage 1 merged model: /content/unsloth_pharma_merge_reload_outputs/stage1_non_instruction_merged_model
Stage 2 adapter: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_adapter
Stage 2 merged model: /content/unsloth_pharma_merge_reload_outputs/stage2_instruction_merged_model
Stage 3 DPO adapter: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_adapter
Final merged model: /content/unsloth_pharma_merge_reload_outputs/stage3_dpo_final_merged_model


### 📌 LEFT vs RIGHT Padding (Important for LLM Training)<br>
#### 🔹 1. RIGHT Padding (Used in SFT / Stage 1 & 2)
##### Format:
> [TEXT TEXT TEXT TEXT PAD PAD PAD]
##### Example:


> Instruction + Response:
"Explain metformin → Metformin is a drug"

> After tokenization:
[Explain metformin Metformin is a drug PAD PAD PAD]

##### Why RIGHT padding is used:


*   Best for causal language modeling (next-token prediction)
*   Keeps real text at the beginning
*   Padding is ignored naturally by attention
*   Stable for training single sequences

##### Used in:
*   Stage 1 (domain pretraining)
*   Stage 2 (instruction SFT)

#### 🔹 2. LEFT Padding (Used in DPO / Stage 3)
##### Format:

> [PAD PAD PAD TEXT TEXT TEXT TEXT]
##### Example:


> Prompt + Response:
"Explain metformin → Metformin is a drug"

> After tokenization:
[PAD PAD PAD Explain metformin Metformin is a drug]

##### Why LEFT padding is used:

*   To compare two sequences token-by-token under the same prompt alignment”
*   Ensures prompt ends at same position across samples
*   Aligns chosen vs rejected sequences properly
*   Required for correct probability comparison in DPO
*   Makes batching stable for pairwise training

##### Used in:
*   Stage 3 (DPO / preference tuning)




| Stage           | Data                       | Trainer      | Model learns                    |
| --------------- | -------------------------- | ------------ | ------------------------------- |
| Non-instruction | Raw PDF paragraphs         | `SFTTrainer` | Domain language and facts       |
| Instruction     | Instruction + response     | `SFTTrainer` | How to answer user instructions |
| Preference/DPO  | Prompt + chosen + rejected | `DPOTrainer` | Which answer is better          |


| Component                    | Source           | Kaam                                            |
| ---------------------------- | ---------------- | ----------------------------------------------- |
| `FastLanguageModel`          | Unsloth          | Fast 4-bit model loading + LoRA patching        |
| `get_peft_model` style logic | Unsloth          | Optimized LoRA training                         |
| `SFTTrainer`                 | Hugging Face TRL | Training loop manage karta hai                  |
| `DPOTrainer`                 | Hugging Face TRL | Preference training loop                        |
| Unsloth patching             | Unsloth          | TRL trainer/model ko faster/low-VRAM banata hai |
